# Explore: the layer-specific figures, scrubbable across all 20 layers

**What this notebook is for.** The report notebooks (02 geometry, 04 parity) each *pin* their
layer-specific figures to one layer: the circumplex at layer 33, the logit lens at 33/57. This
notebook makes those same figures **interactive over the layer axis**. Each figure carries one
Plotly slider. Drag it and watch the plot rebuild at each of the 20 swept layers
`[0, 3, 6, ... , 57]`. Nothing here changes a claim. It is an exploration surface over the
committed data.

**Two tiers, because of where the data lives.**
- *Geometry (free):* the all-layer means bundle is local and holds every layer, so the circumplex is
  recomputed on the fly, with no model and no GPU.
- *Logit lens (needs one pod pass):* reading the lens needs the 31B model's unembedding matrix,
  which is not local. Until `scripts/logit_lens.py --all-layers` runs on the pod, the lens slider
  has only the 2 committed layers (33, 57); after it runs it has all 20. The command is at the
  bottom.

**Acronyms.** *PCA* principal component analysis; *NRC VAD* the NRC valence-arousal-dominance lexicon
(our human reference ratings); *unembedding* the output vocabulary matrix a logit lens reads through.


In [1]:
# this cell loads both models' all-layer means bundles and the NRC VAD lexicon (mirrors 02)
import json
from pathlib import Path

import numpy as np
from sklearn.decomposition import PCA

from emotion_vectors.analysis import load_nrc_vad
from emotion_vectors.artifacts import fetch  # local results/ first, HF otherwise
from emotion_vectors.interactive import layer_scatter_scrubber, layer_table_scrubber

ROOT = Path("..")
IT, BASE = "gemma-4-31b-it", "gemma-4-31b (base)"
bundles = {
    IT: np.load(fetch("emotion_vectors_it_means.npz"), allow_pickle=True),
    BASE: np.load(fetch("emotion_vectors/emotion_means.npz"), allow_pickle=True),
}
emotions = list(map(str, bundles[IT]["emotions"]))
layers = [int(l) for l in bundles[IT]["layers"]]

vad = load_nrc_vad(ROOT / "data/lexicons/NRC-VAD-Lexicon-v2.1/NRC-VAD-Lexicon-v2.1.txt")
matched = [i for i, e in enumerate(emotions) if e.lower() in vad]
valence = np.array([vad[emotions[i].lower()][0] for i in matched])
arousal = np.array([vad[emotions[i].lower()][1] for i in matched])
size = 8 + 22 * (arousal - arousal.min()) / np.ptp(arousal)  # point size = NRC arousal, as in 02
print(
    f"{len(layers)} layers {layers[0]}..{layers[-1]}; NRC matched {len(matched)}/{len(emotions)} emotions"
)

20 layers 0..57; NRC matched 164/171 emotions


## 1. The circumplex, scrubbable across all 20 layers

Free tier: recomputed from the local bundle, with no model needed. Drag the slider to walk the
circumplex down the network. This is the exact figure of notebook 02 section 2, unpinned from layer 33.

In [2]:
# this cell recomputes each model's valence-best x arousal-best plane at EVERY layer and scrubs them
def per_layer_plane(means_all, lp):
    """Same component selection + sign rule as notebook 02 section 2, at one layer."""
    M = means_all[:, lp, :].astype(np.float64)
    pca = PCA(n_components=10)
    scores = pca.fit_transform(M)
    r_val = [np.corrcoef(scores[matched, k], valence)[0, 1] for k in range(10)]
    r_aro = [np.corrcoef(scores[matched, k], arousal)[0, 1] for k in range(10)]
    vb = int(np.argmax(np.abs(r_val)))  # valence-best of the top 10 components
    ab = int(np.argmax([abs(r) if k != vb else -1.0 for k, r in enumerate(r_aro)]))
    if r_val[vb] < 0:  # sign-orient: pleasant to the right, aroused up (same as 02)
        scores[:, vb] *= -1
    if r_aro[ab] < 0:
        scores[:, ab] *= -1
    r_vb = np.corrcoef(scores[matched, vb], valence)[0, 1]
    r_ab = np.corrcoef(scores[matched, ab], arousal)[0, 1]
    evr = pca.explained_variance_ratio_
    return vb, ab, r_vb, r_ab, evr, scores[matched, vb], scores[matched, ab]


panels = []
for label in (IT, BASE):
    means_all = bundles[label]["means"]
    x, y, xt, yt = {}, {}, {}, {}
    for lp, layer in enumerate(layers):
        vb, ab, r_vb, r_ab, evr, xi, yi = per_layer_plane(means_all, lp)
        x[layer], y[layer] = xi, yi
        xt[layer] = f"component {vb + 1} (valence |r|={r_vb:.2f}, {evr[vb]:.0%} var)"
        yt[layer] = f"component {ab + 1} (arousal |r|={r_ab:.2f}, {evr[ab]:.0%} var)"
    panels.append(
        dict(
            name=label,
            x=x,
            y=y,
            xtitle=xt,
            ytitle=yt,
            color=valence,
            size=size,
            text=[emotions[i] for i in matched],
            showscale=(label == BASE),
        )
    )

# anchor: at layer 33 the selection must reproduce notebook 02 section 2 (instruct comp 3, base comp 1)
assert panels[0]["xtitle"][33].split()[1] == "3", panels[0]["xtitle"][33]
assert panels[1]["xtitle"][33].split()[1] == "1", panels[1]["xtitle"][33]
print("anchor OK — L33: instruct valence-best component 3, base component 1 (matches 02 section 2)")

fig = layer_scatter_scrubber(
    layers,
    panels,
    default_layer=33,
    colorbar_title="NRC valence",
    title="Emotion circumplex vs layer — drag the slider "
    "(instruct left / base right; color = NRC valence, size = NRC arousal)"
    "<br><sup>each panel is that model's own valence-best x arousal-best plane at the chosen layer</sup>",
)
fig.show()

anchor OK — L33: instruct valence-best component 3, base component 1 (matches 02 section 2)


<details><summary><b>How to read this figure</b></summary>

Identical construction to notebook 02 section 2, one layer at a time. Each dot is one emotion on that
model's own best affective plane at the chosen layer. The x axis is whichever of the top ten principal
components best correlates with NRC valence. The y axis is whichever of the rest best correlates with
arousal. Each axis is sign-flipped so that pleasant is right and aroused is up. Colour is NRC valence
and size is NRC arousal (the external human ratings), so they stay fixed as you scrub: only the
geometry and the axis labels change. Watch two things down the layers. (1) For `gemma-4-31b`, valence
rides components 1-2 in the mid stack (the published circumplex). (2) For `gemma-4-31b-it`, the valence
axis is demoted to a higher component index with a smaller variance share, so the colour gradient
survives but on a minor component. Weak or scattered instruct planes at some layers are the demotion
signal, not noise.

</details>

## 2. The logit lens, scrubbable across all 20 layers

Pod tier: reading the lens needs the model's unembedding, so this shows all 20 layers only after the
sweep below runs. Until then each slider has the 2 committed layers (33, 57).

In [3]:
# this cell scrubs the logit-lens token tables across layers (all 20 after the sweep; the 2 committed layers until then)
COMMITTED = {
    "it": {33: "logit_lens_it_L33_normed.json", 57: "logit_lens_it_L57.json"},
    "base": {33: "logit_lens_base_L33.json", 57: "logit_lens_base_L57.json"},
}


def lens_layers(key):
    """(layers, {layer: {emotion: {up, down}}}) — the all-layer sweep if present, else the 2 committed layers."""
    all_name = f"logit_lens_{key}_all_layers.json"
    if (ROOT / "results" / all_name).exists():
        d = json.loads(fetch(all_name).read_text())
        return [int(l) for l in d["layers"]], {int(l): t for l, t in d["by_layer"].items()}
    by = {L: json.loads(fetch(p).read_text())["table"] for L, p in COMMITTED[key].items()}
    return sorted(by), by


def lens_columns(by_layer):
    """{layer: [emotion_col, up_col, down_col]} for the table scrubber."""
    return {
        L: [
            list(t),
            [", ".join(v["up"]) for v in t.values()],
            [", ".join(v["down"]) for v in t.values()],
        ]
        for L, t in by_layer.items()
    }


for key, label in (("it", IT), ("base", BASE)):
    lys, by = lens_layers(key)
    default = 57 if key == "it" else 33  # the layer each model is displayed at in notebook 04
    fig = layer_table_scrubber(
        lys,
        lens_columns(by),
        header=["emotion", "top up-weighted tokens", "top down-weighted tokens"],
        default_layer=default if default in lys else lys[0],
        title=f"Logit lens vs layer — {label} (drag the slider)"
        "<br><sup>maps to: Anthropic Table 1; final-RMSNorm scale applied, softcapping ignored</sup>",
    )
    fig.show()
    print(f"{label}: {len(lys)} layer(s) available {lys}")

gemma-4-31b-it: 2 layer(s) available [33, 57]


gemma-4-31b (base): 2 layer(s) available [33, 57]


<details><summary><b>How to read this figure, and how to fill in all 20 layers</b></summary>

Each row is one of the 12 scenario-test emotions, the ones the paper's implicit-emotion scenarios
test. At the chosen layer, a row shows that emotion's logit-lens neighbourhood: the tokens its
vector most up-weights and most down-weights when read through the output vocabulary. The base model
shows affective neighbourhoods (happy toward delightful/wonderful) where the instruct model shows
unrelated fragments. That split is the notebook 04 section 2 finding, here scrubbable by depth.

**To populate all 20 layers**, run the sweep once on the pod (model loaded, ~20 matmuls per emotion),
which writes one small JSON per model that this cell picks up automatically:

```
uv run python scripts/logit_lens.py --all-layers \
    --model google/gemma-4-31b-it --means-bundle results/emotion_vectors_it_means.npz \
    --out results/logit_lens_it_all_layers.json
uv run python scripts/logit_lens.py --all-layers \
    --model google/gemma-4-31b --means-bundle results/emotion_vectors/emotion_means.npz \
    --out results/logit_lens_base_all_layers.json
```

The sweep centres on the mean of those 12 emotions, the same convention as the committed files. It
also self-checks that its layer-33 and layer-57 tables reproduce `logit_lens_*_L33*.json` /
`*_L57.json`. A logged MISMATCH there means the other layers should not be trusted.

</details>